In [ ]:
# In[1]:

import pandas as pd

# Load CSVs
metric_df = pd.read_csv("metric.csv")
trace_df = pd.read_csv("trace.csv")
log_df = pd.read_csv("log.csv")
error_df = pd.read_csv("error_logs.csv")

# Parse timestamps to UTC datetimes (Unix seconds -> UTC)
for df in (metric_df, trace_df, log_df, error_df):
    df['dt'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)

# Helper to build compact summary for files with a "name" column
def build_summary(df, name_col):
    names = []
    if name_col in df.columns:
        names = sorted(df[name_col].dropna().unique())
    sample_names = names[:50]  # limit to at most 50 entries
    summary = pd.DataFrame([{
        "rows": int(df.shape[0]),
        "unique_cmdb_ids": int(df['cmdb_id'].nunique()),
        "min_timestamp_utc": df['dt'].min(),
        "max_timestamp_utc": df['dt'].max(),
        "unique_names_count": int(len(names)),
        "sample_names_limited": sample_names
    }])
    return summary

# Build summaries
metric_summary = build_summary(metric_df, 'kpi_name')
trace_summary = build_summary(trace_df, 'trace_name')
log_summary = build_summary(log_df, 'log_name')

# Error logs summary and up to 10 sample rows (compact)
error_summary = pd.DataFrame([{
    "rows": int(error_df.shape[0]),
    "unique_cmdb_ids": int(error_df['cmdb_id'].nunique()),
    "min_timestamp_utc": error_df['dt'].min(),
    "max_timestamp_utc": error_df['dt'].max()
}])

# Prepare up to 10 sample rows showing timestamp (dt), cmdb_id, message
error_samples = error_df.sort_values('dt')[['dt', 'cmdb_id', 'message']].head(10).reset_index(drop=True)

# Display compact results (use variable names, separated by commas)
metric_summary, trace_summary, log_summary, error_summary, error_samples

```
Out[1]:
```
```python
summary = (
    "Summary of telemetry files (timestamps in UTC):\n\n"
    "metric.csv: 1996 rows, 15 unique cmdb_id, timestamps from 2024-01-19 05:37:00+00:00 "
    "to 2024-01-19 06:01:00+00:00. 8 unique KPI names (examples: cpu, diskio, error, "
    "latency-50, latency-90, ...).\n\n"
    "trace.csv: empty (0 rows, 0 unique cmdb_id, no timestamps, no trace names).\n\n"
    "log.csv: 446 rows, 11 unique cmdb_id, timestamps from 2024-01-19 05:37:00+00:00 "
    "to 2024-01-19 06:01:00+00:00. 2 unique log names: log.error_count, log.row_count.\n\n"
    "error_logs.csv: empty (0 rows, 0 unique cmdb_id, no timestamps, no sample messages)."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(   rows  unique_cmdb_ids         min_timestamp_utc         max_timestamp_utc  unique_names_count                               sample_names_limited
0  1996               15 2024-01-19 05:37:00+00:00 2024-01-19 06:01:00+00:00                   8  [cpu, diskio, error, latency-50, latency-90, m...,    rows  unique_cmdb_ids min_timestamp_utc max_timestamp_utc  unique_names_count sample_names_limited
0     0                0               NaT               NaT                   0                   [],    rows  unique_cmdb_ids         min_timestamp_utc         max_timestamp_utc  unique_names_count              sample_names_limited
0   446               11 2024-01-19 05:37:00+00:00 2024-01-19 06:01:00+00:00                   2  [log.error_count, log.row_count],    rows  unique_cmdb_ids min_timestamp_utc max_timestamp_utc
0     0                0               NaT               NaT, Empty DataFrame
Columns: [dt, cmdb_id, message]
Index: [])```
```

In [ ]:
# In[2]:

import numpy as np
import pandas as pd

# Reuse existing DataFrames: metric_df, log_df (they already have 'dt' parsed)
# If not present, load and parse (safe-guard)
try:
    metric_df
    log_df
except NameError:
    metric_df = pd.read_csv("metric.csv")
    log_df = pd.read_csv("log.csv")
    for df in (metric_df, log_df):
        df['dt'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)

# Incident window (UTC)
start = pd.to_datetime('2024-01-19 05:34:52', utc=True)
end = pd.to_datetime('2024-01-19 06:04:52', utc=True)

def compute_aggregates(df, name_col, source_label):
    # Standardize KPI column name
    df = df.copy()
    df = df.rename(columns={name_col: 'kpi_name'})
    # 1) global percentiles (use entire series per component-kpi)
    gp95 = df.groupby(['cmdb_id','kpi_name'])['value'].quantile(0.95).reset_index().rename(columns={'value':'global_p95'})
    gp05 = df.groupby(['cmdb_id','kpi_name'])['value'].quantile(0.05).reset_index().rename(columns={'value':'global_p5'})
    global_df = gp95.merge(gp05, on=['cmdb_id','kpi_name'])
    # 2) filter to incident window
    win = df[(df['dt'] >= start) & (df['dt'] <= end)].copy()
    if win.shape[0] == 0:
        # No data in window: produce rows for all global groups with zeros/nulls
        out = global_df.copy()
        out['total_points_in_window'] = 0
        out['anomaly_count_high'] = 0
        out['earliest_high_timestamp_utc'] = pd.NaT
        out['max_value_in_window'] = np.nan
        out['anomaly_count_low'] = 0
        out['earliest_low_timestamp_utc'] = pd.NaT
        out['min_value_in_window'] = np.nan
    else:
        # merge global thresholds into window rows to evaluate anomalies
        win = win.merge(global_df, on=['cmdb_id','kpi_name'], how='left')
        win['is_high'] = win['value'] >= win['global_p95']
        win['is_low'] = win['value'] <= win['global_p5']
        # Aggregates within window
        agg = win.groupby(['cmdb_id','kpi_name']).agg(
            total_points_in_window=('value','size'),
            anomaly_count_high=('is_high','sum'),
            earliest_high_timestamp_utc=('dt', lambda s: s[win.loc[s.index,'is_high']].min() if s.shape[0]>0 and win.loc[s.index,'is_high'].any() else pd.NaT),
            max_value_in_window=('value','max'),
            anomaly_count_low=('is_low','sum'),
            earliest_low_timestamp_utc=('dt', lambda s: s[win.loc[s.index,'is_low']].min() if s.shape[0]>0 and win.loc[s.index,'is_low'].any() else pd.NaT),
            min_value_in_window=('value','min')
        ).reset_index()
        # Merge with global to include groups with zero points in window
        out = global_df.merge(agg, on=['cmdb_id','kpi_name'], how='left')
        # Fill NaNs for groups missing in window
        out['total_points_in_window'] = out['total_points_in_window'].fillna(0).astype(int)
        out['anomaly_count_high'] = out['anomaly_count_high'].fillna(0).astype(int)
        # earliest timestamps: keep as NaT where missing
        out['earliest_high_timestamp_utc'] = out['earliest_high_timestamp_utc'].where(out['earliest_high_timestamp_utc'].notna(), pd.NaT)
        out['max_value_in_window'] = out['max_value_in_window'].where(out['max_value_in_window'].notna(), np.nan)
        out['anomaly_count_low'] = out['anomaly_count_low'].fillna(0).astype(int)
        out['earliest_low_timestamp_utc'] = out['earliest_low_timestamp_utc'].where(out['earliest_low_timestamp_utc'].notna(), pd.NaT)
        out['min_value_in_window'] = out['min_value_in_window'].where(out['min_value_in_window'].notna(), np.nan)

    # Add source label
    out['source'] = source_label
    # Keep column order compact
    out = out[['cmdb_id','kpi_name','global_p95','global_p5',
               'total_points_in_window','anomaly_count_high','earliest_high_timestamp_utc','max_value_in_window',
               'anomaly_count_low','earliest_low_timestamp_utc','min_value_in_window','source']]
    return out

metric_res = compute_aggregates(metric_df, 'kpi_name', 'metric')
log_res = compute_aggregates(log_df, 'log_name', 'log')

# Combine results
combined = pd.concat([metric_res, log_res], ignore_index=True)

# Compute severity ratio = max_value_in_window / global_p95 (safe division)
combined['severity_ratio'] = np.where(combined['global_p95'] != 0,
                                      combined['max_value_in_window'] / combined['global_p95'],
                                      np.nan)

# Sort and take top 20 per requirements
combined_sorted = combined.sort_values(
    by=['anomaly_count_high','anomaly_count_low','severity_ratio'],
    ascending=[False, False, False]
).reset_index(drop=True)

# Keep output compact: select columns and top 20 rows
output_cols = ['source','cmdb_id','kpi_name','global_p95','global_p5',
               'total_points_in_window','anomaly_count_high','earliest_high_timestamp_utc','max_value_in_window',
               'anomaly_count_low','earliest_low_timestamp_utc','min_value_in_window','severity_ratio']

combined_head = combined_sorted[output_cols].head(20)

# Display the final compact table
combined_head

```
Out[2]:
```
```python
summary = (
    "Incident window: 2024-01-19 05:34:52 UTC to 2024-01-19 06:04:52 UTC.\n\n"
    "Key findings (compact):\n"
    "- The top anomalies are network/socket-related metrics: 'socket' KPIs for catalogue, catalogue-db, payment, "
    "queue-master, rabbitmq, session-db, user, and user-db each show anomaly_count_high = 25 (all 25 points in-window >= global P95). "
    "Earliest high timestamps for these are 2024-01-19 05:37:00+00:00. These should be investigated first.\n"
    "- rabbitmq-exporter 'mem' also shows anomaly_count_high = 25 (global P95 = 4575232, max in window = 4575232) with earliest 2024-01-19 05:37:00+00:00.\n"
    "- Many other KPIs (various 'error' and 'diskio' metrics and several log.error_count entries) have global P95 = 0 and global P5 = 0, "
    "so all window points equal 0 were counted as anomalies (anomaly_count_high = 25 and anomaly_count_low = 25). These are likely non-actionable/false positives caused by zero-valued baselines and should be treated cautiously.\n\n"
    "Summary recommendation: prioritize investigating the 'socket' spikes across the listed services and the rabbitmq-exporter memory metric. Treat zero-baseline metrics (error/diskio/log.error_count) as low-confidence anomaly signals."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

source            cmdb_id         kpi_name  global_p95  global_p5  total_points_in_window  anomaly_count_high earliest_high_timestamp_utc  max_value_in_window  anomaly_count_low earliest_low_timestamp_utc  min_value_in_window  severity_ratio
0   metric          catalogue           socket        10.0       10.0                      25                  25   2024-01-19 05:37:00+00:00                 10.0                 25  2024-01-19 05:37:00+00:00                 10.0             1.0
1   metric       catalogue-db           socket         4.0        4.0                      25                  25   2024-01-19 05:37:00+00:00                  4.0                 25  2024-01-19 05:37:00+00:00                  4.0             1.0
2   metric            payment           socket         6.0        6.0                      25                  25   2024-01-19 05:37:00+00:00                  6.0                 25  2024-01-19 05:37:00+00:00                  6.0             1.0
3   metric       queue-master           socket         3.0        3.0                      25                  25   2024-01-19 05:37:00+00:00                  3.0                 25  2024-01-19 05:37:00+00:00                  3.0             1.0
4   metric           rabbitmq           socket        11.0       11.0                      25                  25   2024-01-19 05:37:00+00:00                 11.0                 25  2024-01-19 05:37:00+00:00                 11.0             1.0
5   metric  rabbitmq-exporter              mem   4575232.0  4575232.0                      25                  25   2024-01-19 05:37:00+00:00            4575232.0                 25  2024-01-19 05:37:00+00:00            4575232.0             1.0
6   metric  rabbitmq-exporter           socket         1.0        1.0                      25                  25   2024-01-19 05:37:00+00:00                  1.0                 25  2024-01-19 05:37:00+00:00                  1.0             1.0
7   metric         session-db           socket         3.0        3.0                      25                  25   2024-01-19 05:37:00+00:00                  3.0                 25  2024-01-19 05:37:00+00:00                  3.0             1.0
8   metric               user           socket        24.0       24.0                      25                  25   2024-01-19 05:37:00+00:00                 24.0                 25  2024-01-19 05:37:00+00:00                 24.0             1.0
9   metric            user-db           socket        13.0       13.0                      25                  25   2024-01-19 05:37:00+00:00                 13.0                 25  2024-01-19 05:37:00+00:00                 13.0             1.0
10  metric              carts            error         0.0        0.0                      25                  25   2024-01-19 05:37:00+00:00                  0.0                 25  2024-01-19 05:37:00+00:00                  0.0             NaN
11  metric          catalogue           diskio         0.0        0.0                      25                  25   2024-01-19 05:37:00+00:00                  0.0                 25  2024-01-19 05:37:00+00:00                  0.0             NaN
12  metric          catalogue            error         0.0        0.0                      25                  25   2024-01-19 05:37:00+00:00                  0.0                 25  2024-01-19 05:37:00+00:00                  0.0             NaN
13  metric       catalogue-db           diskio         0.0        0.0                      25                  25   2024-01-19 05:37:00+00:00                  0.0                 25  2024-01-19 05:37:00+00:00                  0.0             NaN
14  metric            payment            error         0.0        0.0                      25                  25   2024-01-19 05:37:00+00:00                  0.0                 25  2024-01-19 05:37:00+00:00                  0.0             NaN
15  metric           rabbitmq           diskio         0.0        0.0                      25                  25   2024-01-19 05:37:00+00:00                  0.0                 25  2024-01-19 05:37:00+00:00                  0.0             NaN
16  metric               user            error         0.0        0.0                      25                  25   2024-01-19 05:37:00+00:00                  0.0                 25  2024-01-19 05:37:00+00:00                  0.0             NaN
17     log           carts-db  log.error_count         0.0        0.0                      25                  25   2024-01-19 05:37:00+00:00                  0.0                 25  2024-01-19 05:37:00+00:00                  0.0             NaN
18     log          catalogue  log.error_count         0.0        0.0                      25                  25   2024-01-19 05:37:00+00:00                  0.0                 25  2024-01-19 05:37:00+00:00                  0.0             NaN
19     log          front-end  log.error_count         0.0        0.0                      25                  25   2024-01-19 05:37:00+00:00                  0.0                 25  2024-01-19 05:37:00+00:00                  0.0             NaN```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np

# Reuse metric_df and existing incident window variables if available; otherwise define them
try:
    metric_df
    start, end
except NameError:
    metric_df = pd.read_csv("metric.csv")
    metric_df['dt'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)
    start = pd.to_datetime('2024-01-19 05:34:52', utc=True)
    end = pd.to_datetime('2024-01-19 06:04:52', utc=True)

# Candidate components and filter conditions
candidates = ['catalogue','catalogue-db','payment','queue-master','rabbitmq','session-db','user','user-db','rabbitmq-exporter']

mask_component = metric_df['cmdb_id'].isin(candidates)
mask_socket = metric_df['kpi_name'].str.contains('socket', na=False)
mask_mem_specific = (metric_df['cmdb_id'] == 'rabbitmq-exporter') & (metric_df['kpi_name'] == 'mem')

# Filter rows to candidates and the specified KPI patterns, then to the incident window
filtered = metric_df[ mask_component & (mask_socket | mask_mem_specific) ].copy()
windowed = filtered[(filtered['dt'] >= start) & (filtered['dt'] <= end)].copy()

# If no rows, return empty DataFrame with expected columns
if windowed.shape[0] == 0:
    cols = ['cmdb_id','kpi_name','total_points_in_window','unique_values_count',
            'min_value_in_window','max_value_in_window',
            'first_timestamp_utc','first_value','last_timestamp_utc','last_value','sample_unique_values']
    candidate_summary = pd.DataFrame(columns=cols)
else:
    # Group and compute aggregates
    grp = windowed.groupby(['cmdb_id','kpi_name'])
    total_points = grp.size().rename('total_points_in_window')
    unique_values_count = grp['value'].nunique().rename('unique_values_count')
    min_value = grp['value'].min().rename('min_value_in_window')
    max_value = grp['value'].max().rename('max_value_in_window')
    # idx of first (earliest) and last (latest) timestamps per group
    idx_first = grp['dt'].idxmin().rename('idx_first')
    idx_last = grp['dt'].idxmax().rename('idx_last')
    # sample up to 10 unique values sorted
    sample_unique_values = grp['value'].agg(lambda s: sorted(np.unique(s))[:10]).rename('sample_unique_values')

    agg = pd.concat([total_points, unique_values_count, min_value, max_value, idx_first, idx_last, sample_unique_values], axis=1).reset_index()

    # Extract first/last timestamp and values using the indices
    agg['first_timestamp_utc'] = metric_df.loc[agg['idx_first'], 'dt'].values
    agg['first_value'] = metric_df.loc[agg['idx_first'], 'value'].values
    agg['last_timestamp_utc'] = metric_df.loc[agg['idx_last'], 'dt'].values
    agg['last_value'] = metric_df.loc[agg['idx_last'], 'value'].values

    # Keep only required columns and order them
    candidate_summary = agg[['cmdb_id','kpi_name','total_points_in_window','unique_values_count',
                             'min_value_in_window','max_value_in_window',
                             'first_timestamp_utc','first_value','last_timestamp_utc','last_value','sample_unique_values']]

    # Sort as requested and limit to top 50
    candidate_summary = candidate_summary.sort_values(
        by=['total_points_in_window','max_value_in_window'],
        ascending=[False, False]
    ).reset_index(drop=True).head(50)

# Display the compact summary table
candidate_summary

```
Out[3]:
```
Incident window: 2024-01-19 05:34:52 UTC to 2024-01-19 06:04:52 UTC.

Per (cmdb_id, kpi_name) summary (one row each):

- rabbitmq-exporter — mem:
  total_points=25, unique_values=1, min=4575232.0, max=4575232.0,
  first=2024-01-19 05:37:00+00:00 (4575232.0), last=2024-01-19 06:01:00+00:00 (4575232.0),
  sample_values=[4575232.0]

- user — socket:
  total_points=25, unique_values=1, min=24.0, max=24.0,
  first=2024-01-19 05:37:00+00:00 (24.0), last=2024-01-19 06:01:00+00:00 (24.0),
  sample_values=[24.0]

- user-db — socket:
  total_points=25, unique_values=1, min=13.0, max=13.0,
  first=2024-01-19 05:37:00+00:00 (13.0), last=2024-01-19 06:01:00+00:00 (13.0),
  sample_values=[13.0]

- rabbitmq — socket:
  total_points=25, unique_values=1, min=11.0, max=11.0,
  first=2024-01-19 05:37:00+00:00 (11.0), last=2024-01-19 06:01:00+00:00 (11.0),
  sample_values=[11.0]

- catalogue — socket:
  total_points=25, unique_values=1, min=10.0, max=10.0,
  first=2024-01-19 05:37:00+00:00 (10.0), last=2024-01-19 06:01:00+00:00 (10.0),
  sample_values=[10.0]

- payment — socket:
  total_points=25, unique_values=1, min=6.0, max=6.0,
  first=2024-01-19 05:37:00+00:00 (6.0), last=2024-01-19 06:01:00+00:00 (6.0),
  sample_values=[6.0]

- catalogue-db — socket:
  total_points=25, unique_values=1, min=4.0, max=4.0,
  first=2024-01-19 05:37:00+00:00 (4.0), last=2024-01-19 06:01:00+00:00 (4.0),
  sample_values=[4.0]

- queue-master — socket:
  total_points=25, unique_values=1, min=3.0, max=3.0,
  first=2024-01-19 05:37:00+00:00 (3.0), last=2024-01-19 06:01:00+00:00 (3.0),
  sample_values=[3.0]

- session-db — socket:
  total_points=25, unique_values=1, min=3.0, max=3.0,
  first=2024-01-19 05:37:00+00:00 (3.0), last=2024-01-19 06:01:00+00:00 (3.0),
  sample_values=[3.0]

- rabbitmq-exporter — socket:
  total_points=25, unique_values=1, min=1.0, max=1.0,
  first=2024-01-19 05:37:00+00:00 (1.0), last=2024-01-19 06:01:00+00:00 (1.0),
  sample_values=[1.0]

Interpretation: all listed component/KPI pairs have full coverage in the window (25 points), constant values across the window (unique_values_count = 1), and consistent first/last timestamps at 2024-01-19 05:37:00+00:00 and 2024-01-19 06:01:00+00:00. The most notable absolute magnitude is rabbitmq-exporter.mem = 4,575,232.0 and socket values vary by component (highest socket = 24.0 for user).

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id kpi_name  total_points_in_window  unique_values_count  min_value_in_window  max_value_in_window first_timestamp_utc  first_value  last_timestamp_utc  last_value sample_unique_values
0  rabbitmq-exporter      mem                      25                    1            4575232.0            4575232.0 2024-01-19 05:37:00    4575232.0 2024-01-19 06:01:00   4575232.0          [4575232.0]
1               user   socket                      25                    1                 24.0                 24.0 2024-01-19 05:37:00         24.0 2024-01-19 06:01:00        24.0               [24.0]
2            user-db   socket                      25                    1                 13.0                 13.0 2024-01-19 05:37:00         13.0 2024-01-19 06:01:00        13.0               [13.0]
3           rabbitmq   socket                      25                    1                 11.0                 11.0 2024-01-19 05:37:00         11.0 2024-01-19 06:01:00        11.0               [11.0]
4          catalogue   socket                      25                    1                 10.0                 10.0 2024-01-19 05:37:00         10.0 2024-01-19 06:01:00        10.0               [10.0]
5            payment   socket                      25                    1                  6.0                  6.0 2024-01-19 05:37:00          6.0 2024-01-19 06:01:00         6.0                [6.0]
6       catalogue-db   socket                      25                    1                  4.0                  4.0 2024-01-19 05:37:00          4.0 2024-01-19 06:01:00         4.0                [4.0]
7       queue-master   socket                      25                    1                  3.0                  3.0 2024-01-19 05:37:00          3.0 2024-01-19 06:01:00         3.0                [3.0]
8         session-db   socket                      25                    1                  3.0                  3.0 2024-01-19 05:37:00          3.0 2024-01-19 06:01:00         3.0                [3.0]
9  rabbitmq-exporter   socket                      25                    1                  1.0                  1.0 2024-01-19 05:37:00          1.0 2024-01-19 06:01:00         1.0                [1.0]```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np

# Reuse existing DataFrames and incident window if available, otherwise load and parse
try:
    metric_df
    log_df
    start, end
except NameError:
    metric_df = pd.read_csv("metric.csv")
    log_df = pd.read_csv("log.csv")
    for df in (metric_df, log_df):
        df['dt'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)
    start = pd.to_datetime('2024-01-19 05:34:52', utc=True)
    end = pd.to_datetime('2024-01-19 06:04:52', utc=True)

# Ensure dt exists
if 'dt' not in metric_df.columns:
    metric_df['dt'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)
if 'dt' not in log_df.columns:
    log_df['dt'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True)

# --- Metrics: filter to incident window and compute requested aggregates ---
metric_win = metric_df[(metric_df['dt'] >= start) & (metric_df['dt'] <= end)].copy()

if metric_win.shape[0] == 0:
    metric_result = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','total_points_in_window','unique_values_count',
        'min_value_in_window','max_value_in_window','delta'])
else:
    grp = metric_win.groupby(['cmdb_id','kpi_name'])['value']
    metric_agg = grp.agg(
        total_points_in_window='size',
        unique_values_count=lambda s: s.nunique(),
        min_value_in_window='min',
        max_value_in_window='max'
    ).reset_index()
    metric_agg['delta'] = metric_agg['max_value_in_window'] - metric_agg['min_value_in_window']
    metric_result = metric_agg[
        (metric_agg['unique_values_count'] > 1) | (metric_agg['delta'] > 0)
    ].sort_values(by=['delta','unique_values_count'], ascending=[False, False]).reset_index(drop=True).head(20)

# --- Logs: summarize log.error_count per cmdb_id in the same window ---
log_win = log_df[(log_df['dt'] >= start) & (log_df['dt'] <= end)].copy()
log_error = log_win[ log_win['log_name'] == 'log.error_count' ].copy()

if log_error.shape[0] == 0:
    log_summary = pd.DataFrame(columns=[
        'cmdb_id','count_of_rows_in_window','count_rows_with_value_gt_0','earliest_timestamp_with_value_gt_0'])
else:
    def earliest_gt0(sub):
        s = sub[sub['value'] > 0]
        return s['dt'].min() if not s.empty else pd.NaT

    lg = log_error.groupby('cmdb_id').agg(
        count_of_rows_in_window=('value','size'),
        count_rows_with_value_gt_0=('value', lambda s: int((s > 0).sum()))
    ).reset_index()
    # compute earliest timestamp with value>0
    earliest = log_error.groupby('cmdb_id').apply(earliest_gt0).rename('earliest_timestamp_with_value_gt_0').reset_index()
    log_summary = lg.merge(earliest, on='cmdb_id')
    # Keep only non-empty results and limit to 20 rows
    log_summary = log_summary[ (log_summary['count_of_rows_in_window'] > 0) ].reset_index(drop=True).head(20)

# Display compact outputs
metric_result, log_summary

```
Out[4]:
```
Incident window: 2024-01-19 05:34:52 UTC — 2024-01-19 06:04:52 UTC.

Metrics (rows where values varied in-window; top 20 by delta):
- orders — mem: 25 points, 25 unique values, min=315,160,600, max=639,320,200, delta=324,159,600 (largest change).
- front-end — mem: 25 pts, 25 unique, min≈101,285,900, max≈109,409,500, delta≈8,123,597.
- user-db — mem: 25 pts, 25 unique, delta≈4,898,613.
- carts-db — mem: 25 pts, 25 unique, delta≈4,502,489.
- orders-db — mem: 25 pts, 25 unique, delta≈2,736,002.
- carts-db — diskio: 25 pts, 25 unique, min≈2,551,702, max≈3,822,593, delta≈1,270,891.
- user-db — diskio: 25 pts, 25 unique, delta≈974,445.
- orders-db — diskio: 25 pts, 25 unique, delta≈619,671.
- carts — mem: 25 pts, 25 unique, delta≈361,540.
- session-db — mem: 25 pts, 19 unique, delta≈133,324.
- catalogue — mem: 25 pts, 25 unique, delta≈111,206.
- user — mem: 25 pts, 25 unique, delta≈99,991.
- payment — mem: 25 pts, 25 unique, delta≈86,152.
- queue-master — diskio: 25 pts, 25 unique, delta≈74,997.
- queue-master — mem: 25 pts, 24 unique, delta≈56,593.
- session-db — diskio: 25 pts, 10 unique, delta≈24,630.
- shipping — mem: 25 pts, 21 unique, delta≈15,974.
- rabbitmq — mem: 25 pts, 19 unique, delta≈7,236.
- catalogue-db — mem: 25 pts, 21 unique, delta≈4,096.
- orders — cpu: 25 pts, 25 unique, min≈2.07, max≈50.05, delta≈47.97.

Notes on metrics:
- Most reported series have full coverage in-window (25 points) and many distinct values (unique_values_count ≫ 1), indicating real variability rather than constant readings.
- The largest changes are in memory metrics (notably orders.mem) and notable CPU increase for orders (cpu delta ≈ 48).
- Disk I/O variability is present for several DB and queue components (carts-db, user-db, orders-db, queue-master, session-db).

Logs (log.error_count per cmdb_id in same window):
- No cmdb_id had any log.error_count > 0 in the window. Counts shown are nonzero for rows present (e.g., carts: 20 rows; carts-db: 25; catalogue: 25; front-end: 25; orders: 25; queue-master: 24; user: 25; etc.), but count_rows_with_value_gt_0 = 0 and earliest_timestamp_with_value_gt_0 = None for all.

Bottom line / actionable summary:
- Investigate large memory variability (orders.mem highest) and the orders CPU spike. Also review disk I/O variability across DB and queue services.
- Log error counts do not show increases during the window (no error_count > 0), so metric anomalies are not accompanied by log.error_count spikes.

The original code execution output of IPython Kernel is also provided below for reference:

(         cmdb_id kpi_name  total_points_in_window  unique_values_count  min_value_in_window  max_value_in_window         delta
0         orders      mem                      25                   25         3.151606e+08         6.393202e+08  3.241596e+08
1      front-end      mem                      25                   25         1.012859e+08         1.094095e+08  8.123597e+06
2        user-db      mem                      25                   25         2.169656e+08         2.218642e+08  4.898613e+06
3       carts-db      mem                      25                   25         2.293320e+08         2.338345e+08  4.502489e+06
4      orders-db      mem                      25                   25         1.194972e+08         1.222332e+08  2.736002e+06
5       carts-db   diskio                      25                   25         2.551702e+06         3.822593e+06  1.270891e+06
6        user-db   diskio                      25                   25         2.110592e+06         3.085037e+06  9.744453e+05
7      orders-db   diskio                      25                   25         2.664560e+05         8.861275e+05  6.196715e+05
8          carts      mem                      25                   25         2.221343e+08         2.224958e+08  3.615403e+05
9     session-db      mem                      25                   19         2.889523e+06         3.022848e+06  1.333248e+05
10     catalogue      mem                      25                   25         6.306338e+06         6.417545e+06  1.112064e+05
11          user      mem                      25                   25         8.899603e+06         8.999595e+06  9.999135e+04
12       payment      mem                      25                   25         4.734839e+06         4.820992e+06  8.615253e+04
13  queue-master   diskio                      25                   25         1.336681e+04         8.836411e+04  7.499730e+04
14  queue-master      mem                      25                   24         2.683372e+08         2.683937e+08  5.659307e+04
15    session-db   diskio                      25                   10         0.000000e+00         2.463019e+04  2.463019e+04
16      shipping      mem                      25                   21         3.064914e+08         3.065074e+08  1.597440e+04
17      rabbitmq      mem                      25                   19         9.845064e+07         9.845787e+07  7.236267e+03
18  catalogue-db      mem                      25                   21         4.324352e+08         4.324393e+08  4.096000e+03
19        orders      cpu                      25                   25         2.074444e+00         5.004670e+01  4.797226e+01,          cmdb_id  count_of_rows_in_window  count_rows_with_value_gt_0 earliest_timestamp_with_value_gt_0
0          carts                       20                           0                                NaT
1       carts-db                       25                           0                                NaT
2      catalogue                       25                           0                                NaT
3      front-end                       25                           0                                NaT
4         orders                       25                           0                                NaT
5      orders-db                        1                           0                                NaT
6        payment                       25                           0                                NaT
7   queue-master                       24                           0                                NaT
8     session-db                        4                           0                                NaT
9       shipping                       24                           0                                NaT
10          user                       25                           0                                NaT)```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np

# Reuse existing metric_df and incident window if available; otherwise load and parse
try:
    metric_df
    start, end
except NameError:
    metric_df = pd.read_csv("metric.csv")
    metric_df['dt'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)
    start = pd.to_datetime('2024-01-19 05:34:52', utc=True)
    end = pd.to_datetime('2024-01-19 06:04:52', utc=True)

# Ensure dt column exists and timestamps parsed
if 'dt' not in metric_df.columns:
    metric_df['dt'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)

# Target components
targets = ["carts","catalogue","orders","payment","user"]

# 1) Compute global percentiles (P95 and P5) using the full series for these components
full_subset = metric_df[metric_df['cmdb_id'].isin(targets)].copy()

# If no data, return empty DataFrame with expected columns
if full_subset.empty:
    result = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','global_p95','global_p5','total_points_in_window',
        'anomaly_count_high','earliest_high_timestamp_utc','max_value_in_window',
        'anomaly_count_low','earliest_low_timestamp_utc','min_value_in_window','severity_ratio'
    ])
else:
    gp95 = full_subset.groupby(['cmdb_id','kpi_name'])['value'].quantile(0.95).rename('global_p95').reset_index()
    gp05 = full_subset.groupby(['cmdb_id','kpi_name'])['value'].quantile(0.05).rename('global_p5').reset_index()
    thresholds = gp95.merge(gp05, on=['cmdb_id','kpi_name'])

    # 2) Filter original rows to the incident window for the target components
    windowed = metric_df[
        (metric_df['dt'] >= start) & (metric_df['dt'] <= end) & (metric_df['cmdb_id'].isin(targets))
    ].copy()

    # Merge thresholds into windowed rows (so each row has its group's global p95/p5)
    windowed = windowed.merge(thresholds, on=['cmdb_id','kpi_name'], how='left')

    # 3) For each (cmdb_id,kpi_name) compute requested aggregates
    def summarize_group(g):
        total = int(g.shape[0])
        # global thresholds (should be same for group)
        gp95_val = g['global_p95'].iloc[0]
        gp05_val = g['global_p5'].iloc[0]
        # anomaly counts
        high_mask = g['value'] >= gp95_val
        low_mask = g['value'] <= gp05_val
        an_high = int(high_mask.sum())
        an_low = int(low_mask.sum())
        earliest_high = g.loc[high_mask, 'dt'].min() if an_high > 0 else pd.NaT
        earliest_low = g.loc[low_mask, 'dt'].min() if an_low > 0 else pd.NaT
        maxv = g['value'].max() if total>0 else np.nan
        minv = g['value'].min() if total>0 else np.nan
        return pd.Series({
            'global_p95': gp95_val,
            'global_p5': gp05_val,
            'total_points_in_window': total,
            'anomaly_count_high': an_high,
            'earliest_high_timestamp_utc': earliest_high,
            'max_value_in_window': maxv,
            'anomaly_count_low': an_low,
            'earliest_low_timestamp_utc': earliest_low,
            'min_value_in_window': minv
        })

    if windowed.shape[0] == 0:
        # No data in window -> empty result
        result = pd.DataFrame(columns=[
            'cmdb_id','kpi_name','global_p95','global_p5','total_points_in_window',
            'anomaly_count_high','earliest_high_timestamp_utc','max_value_in_window',
            'anomaly_count_low','earliest_low_timestamp_utc','min_value_in_window','severity_ratio'
        ])
    else:
        grouped = windowed.groupby(['cmdb_id','kpi_name']).apply(summarize_group).reset_index()

        # 4) Compute severity_ratio
        grouped['severity_ratio'] = np.where(grouped['global_p95'] > 0,
                                             grouped['max_value_in_window'] / grouped['global_p95'],
                                             np.nan)

        # Keep only rows where anomaly_count_high>0 or anomaly_count_low>0
        filtered = grouped[(grouped['anomaly_count_high'] > 0) | (grouped['anomaly_count_low'] > 0)].copy()

        # Sort and limit to top 20
        result = filtered.sort_values(by=['anomaly_count_high','severity_ratio'], ascending=[False, False]).reset_index(drop=True).head(20)

# Keep columns order compact
result = result[[
    'cmdb_id','kpi_name','global_p95','global_p5','total_points_in_window',
    'anomaly_count_high','earliest_high_timestamp_utc','max_value_in_window',
    'anomaly_count_low','earliest_low_timestamp_utc','min_value_in_window','severity_ratio'
]]

result

```
Out[5]:
```
```python
summary = (
    "Incident window: 2024-01-19 05:34:52 UTC — 2024-01-19 06:04:52 UTC.\n\n"
    "Key findings for carts, catalogue, orders, payment, user (metric telemetry):\n\n"
    "1) Persistent socket anomalies (highest priority):\n"
    "- catalogue | socket: global P95=10.0, P5=10.0, 25 points in-window, anomaly_count_high=25 (earliest 2024-01-19 05:37:00 UTC), max=10.0, severity_ratio=1.00\n"
    "- payment   | socket: global P95=6.0, P5=6.0, 25 pts, anomaly_count_high=25 (earliest 2024-01-19 05:37:00 UTC), max=6.0, severity_ratio=1.00\n"
    "- user      | socket: global P95=24.0, P5=24.0, 25 pts, anomaly_count_high=25 (earliest 2024-01-19 05:37:00 UTC), max=24.0, severity_ratio=1.00\n"
    "These three services have all in-window socket values at or above their historical P95 starting ~05:37 UTC — investigate socket/network issues first.\n\n"
    "2) Orders shows actionable anomalies (smaller count but higher severity):\n"
    "- orders | error:   global P95≈4.6489, P5=0.0; 25 pts, anomaly_count_high=2 (earliest 05:37 UTC), max≈10.966, severity≈2.36\n"
    "- orders | workload: global P95≈5.8597, P5≈1.6707; 25 pts, anomaly_count_high=2 (earliest 05:37 UTC), max≈10.966, severity≈1.87\n"
    "=> Orders has a few high outliers (both error and workload) with notable severity ratios; prioritize checking Orders traces/logs and its recent spikes.\n\n"
    "3) Other metric anomalies (many zero-baseline signals — low confidence):\n"
    "- carts/catalogue/payment/user contain several 'error' and 'diskio' series where global P95 and P5 = 0.0, and all window points = 0.0 were flagged as both high and low anomalies. These are likely false/low-confidence signals caused by zero baselines and should be treated cautiously.\n\n"
    "4) Additional lower-priority signals (few points):\n"
    "- Various latency and CPU KPIs across carts, orders, user, payment, catalogue show anomaly_count_high=2 each (small number of outliers). Example: carts latency-50, orders latency-50/90, carts/user/payment CPU — investigate if corroborated by traces.\n\n"
    "Overall recommendation: First investigate socket/network issues for catalogue, payment, and user (these are full-window P95 breaches). Next, investigate Orders (error/workload spikes and CPU/latency outliers). Treat zero-valued error/diskio metrics as low-confidence and deprioritize unless other evidence supports them."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id    kpi_name  global_p95  global_p5  total_points_in_window  anomaly_count_high earliest_high_timestamp_utc  max_value_in_window  anomaly_count_low earliest_low_timestamp_utc  min_value_in_window  severity_ratio
0   catalogue      socket   10.000000  10.000000                      25                  25   2024-01-19 05:37:00+00:00            10.000000                 25  2024-01-19 05:37:00+00:00            10.000000        1.000000
1     payment      socket    6.000000   6.000000                      25                  25   2024-01-19 05:37:00+00:00             6.000000                 25  2024-01-19 05:37:00+00:00             6.000000        1.000000
2        user      socket   24.000000  24.000000                      25                  25   2024-01-19 05:37:00+00:00            24.000000                 25  2024-01-19 05:37:00+00:00            24.000000        1.000000
3       carts       error    0.000000   0.000000                      25                  25   2024-01-19 05:37:00+00:00             0.000000                 25  2024-01-19 05:37:00+00:00             0.000000             NaN
4   catalogue      diskio    0.000000   0.000000                      25                  25   2024-01-19 05:37:00+00:00             0.000000                 25  2024-01-19 05:37:00+00:00             0.000000             NaN
5   catalogue       error    0.000000   0.000000                      25                  25   2024-01-19 05:37:00+00:00             0.000000                 25  2024-01-19 05:37:00+00:00             0.000000             NaN
6     payment       error    0.000000   0.000000                      25                  25   2024-01-19 05:37:00+00:00             0.000000                 25  2024-01-19 05:37:00+00:00             0.000000             NaN
7        user       error    0.000000   0.000000                      25                  25   2024-01-19 05:37:00+00:00             0.000000                 25  2024-01-19 05:37:00+00:00             0.000000             NaN
8      orders       error    4.648920   0.000000                      25                   2   2024-01-19 05:37:00+00:00            10.966375                 23  2024-01-19 05:39:00+00:00             0.000000        2.358908
9      orders    workload    5.859697   1.670685                      25                   2   2024-01-19 05:37:00+00:00            10.966375                  2  2024-01-19 05:51:00+00:00             1.633267        1.871492
10      carts  latency-50    0.860877   0.380730                      25                   2   2024-01-19 05:41:00+00:00             1.329767                  2  2024-01-19 05:44:00+00:00             0.286180        1.544665
11     orders  latency-50    0.746069   0.030578                      25                   2   2024-01-19 05:52:00+00:00             1.086763                  2  2024-01-19 05:37:00+00:00             0.000341        1.456652
12      carts         cpu    1.717726   1.294506                      25                   2   2024-01-19 05:50:00+00:00             2.405804                  2  2024-01-19 05:37:00+00:00             1.281598        1.400575
13     orders  latency-90    2.002096   0.288296                      25                   2   2024-01-19 05:54:00+00:00             2.558821                  2  2024-01-19 05:37:00+00:00             0.056881        1.278071
14       user         cpu    1.285173   0.902296                      25                   2   2024-01-19 05:37:00+00:00             1.518344                  2  2024-01-19 05:53:00+00:00             0.864782        1.181431
15       user    workload   19.751487  15.482300                      25                   2   2024-01-19 05:37:00+00:00            21.433000                  2  2024-01-19 05:51:00+00:00            14.751117        1.085134
16      carts      socket   47.966667  32.166667                      25                   2   2024-01-19 05:54:00+00:00            51.200000                  2  2024-01-19 05:37:00+00:00            30.633333        1.067408
17       user  latency-90    0.016692   0.008771                      25                   2   2024-01-19 05:55:00+00:00             0.017754                  2  2024-01-19 05:44:00+00:00             0.008752        1.063597
18    payment         cpu    0.081581   0.049641                      25                   2   2024-01-19 05:45:00+00:00             0.086053                  2  2024-01-19 05:37:00+00:00             0.038296        1.054809
19  catalogue         cpu    0.172433   0.149478                      25                   2   2024-01-19 05:37:00+00:00             0.181054                  2  2024-01-19 05:48:00+00:00             0.148241        1.049999```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np

# Reuse metric_df and incident window if available; otherwise load and parse
try:
    metric_df
    start, end
except NameError:
    metric_df = pd.read_csv("metric.csv")
    metric_df['dt'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)
    start = pd.to_datetime('2024-01-19 05:34:52', utc=True)
    end = pd.to_datetime('2024-01-19 06:04:52', utc=True)

# Ensure dt exists
if 'dt' not in metric_df.columns:
    metric_df['dt'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)

targets = ["carts","catalogue","orders","payment","user"]

# 1) compute global p95 and p5 for all (cmdb_id,kpi_name) for the target components using full series
full_subset = metric_df[ metric_df['cmdb_id'].isin(targets) ].copy()
if full_subset.empty:
    segments_df = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','breach_type','segment_start_timestamp_utc','segment_end_timestamp_utc',
        'length_in_points','max_value_in_segment','min_value_in_segment','global_p95','global_p5','severity_ratio'
    ])
else:
    gp95 = full_subset.groupby(['cmdb_id','kpi_name'])['value'].quantile(0.95).rename('global_p95').reset_index()
    gp05 = full_subset.groupby(['cmdb_id','kpi_name'])['value'].quantile(0.05).rename('global_p5').reset_index()
    thresholds = gp95.merge(gp05, on=['cmdb_id','kpi_name'])

    # 2) filter rows to incident window for the targets
    windowed = metric_df[
        (metric_df['dt'] >= start) & (metric_df['dt'] <= end) & (metric_df['cmdb_id'].isin(targets))
    ].copy()
    # sort for reliable consecutive detection
    windowed = windowed.sort_values(['cmdb_id','kpi_name','dt']).reset_index(drop=True)

    # merge thresholds into windowed (so each row has group's global thresholds)
    windowed = windowed.merge(thresholds, on=['cmdb_id','kpi_name'], how='left')

    segments = []
    # helper to scan consecutive breaches per group
    for (cmdb,kpi), grp in windowed.groupby(['cmdb_id','kpi_name'], sort=False):
        if grp.empty:
            continue
        grp = grp.sort_values('dt').reset_index(drop=True)
        # compute flags
        gp95_val = grp['global_p95'].iloc[0]
        gp05_val = grp['global_p5'].iloc[0]
        high_mask = grp['value'] >= gp95_val
        low_mask = grp['value'] <= gp05_val
        dt_vals = grp['dt'].values
        val_vals = grp['value'].values

        # function to extract segments for a given boolean mask and breach_type
        def extract_segments(mask, breach_type):
            seg_start_idx = None
            seg_end_idx = None
            seg_len = 0
            for i in range(len(mask)):
                if mask.iloc[i]:
                    if seg_start_idx is None:
                        # start new segment
                        seg_start_idx = i
                        seg_end_idx = i
                        seg_len = 1
                    else:
                        # check if consecutive in time: previous dt + 60s == current dt
                        prev_dt = dt_vals[i-1]
                        cur_dt = dt_vals[i]
                        gap = (pd.to_datetime(cur_dt) - pd.to_datetime(prev_dt)).total_seconds()
                        if gap == 60 and mask.iloc[i-1]:
                            seg_end_idx = i
                            seg_len += 1
                        else:
                            # end previous, start new
                            if seg_len >= 2:
                                seg_slice = grp.iloc[seg_start_idx:seg_end_idx+1]
                                segments.append({
                                    'cmdb_id': cmdb,
                                    'kpi_name': kpi,
                                    'breach_type': breach_type,
                                    'segment_start_timestamp_utc': seg_slice['dt'].iloc[0],
                                    'segment_end_timestamp_utc': seg_slice['dt'].iloc[-1],
                                    'length_in_points': seg_len,
                                    'max_value_in_segment': seg_slice['value'].max(),
                                    'min_value_in_segment': seg_slice['value'].min(),
                                    'global_p95': gp95_val,
                                    'global_p5': gp05_val
                                })
                            seg_start_idx = i
                            seg_end_idx = i
                            seg_len = 1
                else:
                    # mask false, if previous segment open, close it
                    if seg_start_idx is not None:
                        if seg_len >= 2:
                            seg_slice = grp.iloc[seg_start_idx:seg_end_idx+1]
                            segments.append({
                                'cmdb_id': cmdb,
                                'kpi_name': kpi,
                                'breach_type': breach_type,
                                'segment_start_timestamp_utc': seg_slice['dt'].iloc[0],
                                'segment_end_timestamp_utc': seg_slice['dt'].iloc[-1],
                                'length_in_points': seg_len,
                                'max_value_in_segment': seg_slice['value'].max(),
                                'min_value_in_segment': seg_slice['value'].min(),
                                'global_p95': gp95_val,
                                'global_p5': gp05_val
                            })
                        seg_start_idx = None
                        seg_end_idx = None
                        seg_len = 0
            # after loop, close open segment
            if seg_start_idx is not None and seg_len >= 2:
                seg_slice = grp.iloc[seg_start_idx:seg_end_idx+1]
                segments.append({
                    'cmdb_id': cmdb,
                    'kpi_name': kpi,
                    'breach_type': breach_type,
                    'segment_start_timestamp_utc': seg_slice['dt'].iloc[0],
                    'segment_end_timestamp_utc': seg_slice['dt'].iloc[-1],
                    'length_in_points': seg_len,
                    'max_value_in_segment': seg_slice['value'].max(),
                    'min_value_in_segment': seg_slice['value'].min(),
                    'global_p95': gp95_val,
                    'global_p5': gp05_val
                })

        # extract high and low segments separately
        if high_mask.any():
            extract_segments(high_mask, 'high')
        if low_mask.any():
            extract_segments(low_mask, 'low')

    # build DataFrame
    if len(segments) == 0:
        segments_df = pd.DataFrame(columns=[
            'cmdb_id','kpi_name','breach_type','segment_start_timestamp_utc','segment_end_timestamp_utc',
            'length_in_points','max_value_in_segment','min_value_in_segment','global_p95','global_p5','severity_ratio'
        ])
    else:
        segments_df = pd.DataFrame(segments)
        # compute severity_ratio for high breaches only
        def compute_sev(row):
            if row['breach_type'] == 'high':
                return (row['max_value_in_segment'] / row['global_p95']) if row['global_p95'] and row['global_p95'] > 0 else np.nan
            else:
                return np.nan
        segments_df['severity_ratio'] = segments_df.apply(compute_sev, axis=1)

        # Keep only segments length >=2 already ensured, sort by severity_ratio desc then length desc, top 20
        segments_df = segments_df.sort_values(by=['severity_ratio','length_in_points'], ascending=[False, False]).reset_index(drop=True).head(20)

# Final compact output
segments_df

```
Out[6]:
```
```python
summary = (
    "Incident window: 2024-01-19 05:34:52 UTC — 2024-01-19 06:04:52 UTC.\n\n"
    "Top sustained breach findings (per detected segments):\n"
    "- Long, full-window HIGH socket breaches (sustained, 25 points): catalogue, payment, user — all socket values >= their historical P95 from ~05:37:00 through 06:01:00. These are highest-priority sustained issues.\n"
    "- Short but higher-severity HIGH bursts (length=2 points) around 05:37–05:39 UTC in Orders: \n"
    "  * orders | error (severity ≈ 2.36)\n"
    "  * orders | workload (severity ≈ 1.87)\n"
    "  * orders | socket (severity ≈ 1.04)\n"
    "  * orders | mem (severity ≈ 1.01)\n"
    "  These indicate brief but notable spikes in Orders' errors, workload, memory and socket metrics.\n"
    "- Short HIGH bursts (length=2) for user: cpu and workload (severity ≈ 1.18 and 1.09) at ~05:37–05:38 UTC.\n"
    "- Several other segments exist (e.g., carts mem, payment workload) with small severity (>1.0) but are shorter (2 points).\n"
    "- Many detected 'high' or 'low' segments for error/diskio across services have global thresholds of 0 and produce NaN severity; these are low-confidence signals (zero-baseline) and likely not actionable alone.\n\n"
    "Actionable conclusions:\n"
    "1) Immediately investigate network/socket issues for catalogue, payment, and user — these are sustained (full-window) breaches at/above P95 and are the highest-priority signals.\n"
    "2) Investigate Orders for short but higher-severity spikes (errors, workload, mem, socket) occurring ~05:37–05:39 UTC; correlate with traces/logs and upstream traffic.\n"
    "3) Check user service CPU/workload short spikes next.\n"
    "4) Deprioritize zero-baseline error/diskio signals unless corroborated by other evidence.\n\n"
    "Overall root-cause focus: start with network/socket problems affecting multiple services, then drill into Orders for the observed brief resource/error spikes."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id  kpi_name breach_type segment_start_timestamp_utc segment_end_timestamp_utc  length_in_points  max_value_in_segment  min_value_in_segment    global_p95     global_p5  severity_ratio
0      orders     error        high   2024-01-19 05:37:00+00:00 2024-01-19 05:38:00+00:00                 2          1.096637e+01          5.811150e+00  4.648920e+00  0.000000e+00        2.358908
1      orders  workload        high   2024-01-19 05:37:00+00:00 2024-01-19 05:38:00+00:00                 2          1.096637e+01          6.816583e+00  5.859697e+00  1.670685e+00        1.871492
2        user       cpu        high   2024-01-19 05:37:00+00:00 2024-01-19 05:38:00+00:00                 2          1.518344e+00          1.352281e+00  1.285173e+00  9.022957e-01        1.181431
3        user  workload        high   2024-01-19 05:37:00+00:00 2024-01-19 05:38:00+00:00                 2          2.143300e+01          2.021658e+01  1.975149e+01  1.548230e+01        1.085134
4      orders    socket        high   2024-01-19 05:38:00+00:00 2024-01-19 05:39:00+00:00                 2          4.258333e+01          4.143333e+01  4.076667e+01  1.842000e+01        1.044563
5      orders       mem        high   2024-01-19 05:38:00+00:00 2024-01-19 05:39:00+00:00                 2          6.393202e+08          6.370769e+08  6.323711e+08  3.162649e+08        1.010989
6     payment  workload        high   2024-01-19 05:49:00+00:00 2024-01-19 05:50:00+00:00                 2          1.975567e+00          1.974450e+00  1.971343e+00  1.152120e+00        1.002142
7       carts       mem        high   2024-01-19 05:54:00+00:00 2024-01-19 05:55:00+00:00                 2          2.224958e+08          2.223159e+08  2.223124e+08  2.221490e+08        1.000825
8   catalogue    socket        high   2024-01-19 05:37:00+00:00 2024-01-19 06:01:00+00:00                25          1.000000e+01          1.000000e+01  1.000000e+01  1.000000e+01        1.000000
9     payment    socket        high   2024-01-19 05:37:00+00:00 2024-01-19 06:01:00+00:00                25          6.000000e+00          6.000000e+00  6.000000e+00  6.000000e+00        1.000000
10       user    socket        high   2024-01-19 05:37:00+00:00 2024-01-19 06:01:00+00:00                25          2.400000e+01          2.400000e+01  2.400000e+01  2.400000e+01        1.000000
11      carts     error        high   2024-01-19 05:37:00+00:00 2024-01-19 06:01:00+00:00                25          0.000000e+00          0.000000e+00  0.000000e+00  0.000000e+00             NaN
12      carts     error         low   2024-01-19 05:37:00+00:00 2024-01-19 06:01:00+00:00                25          0.000000e+00          0.000000e+00  0.000000e+00  0.000000e+00             NaN
13  catalogue    diskio        high   2024-01-19 05:37:00+00:00 2024-01-19 06:01:00+00:00                25          0.000000e+00          0.000000e+00  0.000000e+00  0.000000e+00             NaN
14  catalogue    diskio         low   2024-01-19 05:37:00+00:00 2024-01-19 06:01:00+00:00                25          0.000000e+00          0.000000e+00  0.000000e+00  0.000000e+00             NaN
15  catalogue     error        high   2024-01-19 05:37:00+00:00 2024-01-19 06:01:00+00:00                25          0.000000e+00          0.000000e+00  0.000000e+00  0.000000e+00             NaN
16  catalogue     error         low   2024-01-19 05:37:00+00:00 2024-01-19 06:01:00+00:00                25          0.000000e+00          0.000000e+00  0.000000e+00  0.000000e+00             NaN
17  catalogue    socket         low   2024-01-19 05:37:00+00:00 2024-01-19 06:01:00+00:00                25          1.000000e+01          1.000000e+01  1.000000e+01  1.000000e+01             NaN
18    payment     error        high   2024-01-19 05:37:00+00:00 2024-01-19 06:01:00+00:00                25          0.000000e+00          0.000000e+00  0.000000e+00  0.000000e+00             NaN
19    payment     error         low   2024-01-19 05:37:00+00:00 2024-01-19 06:01:00+00:00                25          0.000000e+00          0.000000e+00  0.000000e+00  0.000000e+00             NaN```
```